# Model Evaluation

In [1]:
from pyspark.sql import SparkSession

In [10]:
spark = (
    SparkSession.builder
    .appName("BusServiceReliability")
    .master("local[*]")
    .getOrCreate()
)

In [16]:
from pyspark.ml.classification import DecisionTreeClassificationModel

model = DecisionTreeClassificationModel.load(
    "../models/decision_tree_model"
)

In [17]:
df = spark.read.parquet("../outputs/cleaned_timetable_parquet")

## Prepare Dataset

In [18]:
from pyspark.sql.functions import when, col
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Create the exact same target variable
df = df.withColumn(
    "target",
    when(col("route_size") == "Long", 1).otherwise(0)
)

# 2. Use the exact same indexer (service_code, NOT operator)
indexer = StringIndexer(
    inputCol="service_code",
    outputCol="service_code_index"
)
df = indexer.fit(df).transform(df)

# 3. Use the exact same VectorAssembler inputs
assembler = VectorAssembler(
    inputCols=[
        "stop_sequence",
        "service_code_index"
    ],
    outputCol="features"
)
dataset = assembler.transform(df)

# 4. Use the exact same split with the same seed
train_data, test_data = dataset.randomSplit([0.8, 0.2], seed=42)

## Generate Predictions

In [19]:
predictions = model.transform(test_data)

## Model Accuracy

In [20]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import pandas as pd

# 1. Set up all Evaluators
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="accuracy")
evaluator_precision = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="f1")
evaluator_roc = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# 2. Calculate metrics for the Decision Tree predictions
dt_metrics = {
    "Model": "Decision Tree",
    "Accuracy": evaluator_accuracy.evaluate(predictions),
    "Precision": evaluator_precision.evaluate(predictions),
    "Recall": evaluator_recall.evaluate(predictions),
    "F1-score": evaluator_f1.evaluate(predictions),
    "ROC-AUC": evaluator_roc.evaluate(predictions)
}

# 3. Convert to Pandas DataFrame for a nice table
metrics_table = [dt_metrics]
results_df = pd.DataFrame(metrics_table)

# Format numbers to 4 decimal places
for col in ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]:
    results_df[col] = results_df[col].round(4)

print("\n✅ ✅ ✅ DECISION TREE EVALUATION METRICS ✅ ✅ ✅\n")
display(results_df)


✅ ✅ ✅ DECISION TREE EVALUATION METRICS ✅ ✅ ✅



,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Decision Tree,0.8611,0.7416,0.8611,0.7969,0.5


## Display Predictions

In [21]:
predictions.select(
    "route_size",
    "target",
    "prediction",
    "probability"
).show(10, truncate=False)

+----------+------+----------+-----------+
|route_size|target|prediction|probability|
+----------+------+----------+-----------+
|Long      |1     |0.0       |[1.0,0.0]  |
|Long      |1     |0.0       |[1.0,0.0]  |
|Long      |1     |0.0       |[1.0,0.0]  |
|Long      |1     |0.0       |[1.0,0.0]  |
|Short     |0     |0.0       |[1.0,0.0]  |
|Short     |0     |0.0       |[1.0,0.0]  |
|Short     |0     |0.0       |[1.0,0.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
+----------+------+----------+-----------+
only showing top 10 rows


## Prediction Distribution

In [22]:
predictions.groupBy("prediction").count().show()

[Stage 60:=======================>                                 (5 + 7) / 12]

+----------+-----+
|prediction|count|
+----------+-----+
|       0.0|53919|
+----------+-----+



## Summary

The Decision Tree model was successfully evaluated using the testing dataset. The model generated predictions for unseen data and achieved a measurable classification accuracy. The evaluation results indicate that the model can distinguish between long and non-long routes based on the selected features. These findings demonstrate the effectiveness of Spark MLlib for building scalable machine learning pipelines on timetable data.